# AI Fairness Dashboard

This notebook **showcases** the end-to-end pipeline without replacing the production scripts. Run it after `run_pipeline.py` (or with existing `artifacts/`) so tables and figures load from saved outputs.

**Setup:** open this folder from the repo root, kernel using the project venv (`./venv`), and set the working directory to the repository root (see first code cell).

## 1. What we built

- **Deterministic audit:** standard classifiers → fairness metrics (Python / AIF360) → rule-based severity, causes, mitigations → Semantic Scholar evidence.
- **LLM benchmark (Gemini / OpenAI):** same Python-computed metrics + evidence → qualitative audit and refinement cycles → scored against a shared reference spec.
- **Gold-standard direction:** a project-defined rubric (metrics + narrative + evidence + mitigations); our pipeline is one system evaluated against it, alongside LLMs.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Markdown, display, Image

# Repo root: notebooks/ -> parent is repo
REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent
ART = REPO / "artifacts"
assert ART.is_dir(), f"Expected {ART}. Run pipeline from repo root or cd to repo root."
print("Repo:", REPO)


def show_markdown_excerpt(path: Path, heading: str, max_chars: int = 4500) -> None:
    """Render a truncated markdown file for stakeholder viewing."""
    if not path.exists():
        display(Markdown(f"### {heading}\n\n*File not found:* `{path}`"))
        return
    text = path.read_text(encoding="utf-8")
    body = text[:max_chars] + ("\n\n*(truncated…)*" if len(text) > max_chars else "")
    display(Markdown(f"### {heading}\n\n{body}"))

## 2. Pipeline flow (chronological)

1. Clean → train (`RandomForest` / `LogisticRegression` per dataset scripts) → `classification_predictions.csv`  
2. `compute_fairness.py` → `fairness_metrics.csv`  
3. `qualitative_analysis.py` → `qualitative_report.md` + `qualitative_research_evidence.json`  
4. Optional: `llm_fairness_analysis.py` / `openai_fairness_analysis.py` → provider folders under `metrics/fairness/` (mirrored in `artifacts/`)

Orchestrator: `run_pipeline.py` at repo root.

## 3. Deterministic fairness metrics (German Credit & HMDA)

Loaded from consolidated artifacts after a full run.

In [ ]:
from IPython.display import display

for ds in ["german_credit", "hmda"]:
    p = ART / ds / "fairness" / "fairness_metrics.csv"
    if not p.exists():
        print(f"Missing {p}")
        continue
    print(f"\n=== {ds} ===")
    display(pd.read_csv(p))

## 3b. Our deterministic model — full qualitative audit

This is **our own pipeline output** (`scripts/qualitative_analysis.py`): rule-based severity, root causes, mitigations, and Semantic Scholar evidence. Same metrics as in §3.

Below: excerpts from `qualitative_report.md` per dataset (full files live under `artifacts/{dataset}/fairness/`).

In [ ]:
for ds, label in [("german_credit", "German Credit"), ("hmda", "HMDA")]:
    show_markdown_excerpt(
        ART / ds / "fairness" / "qualitative_report.md",
        f"Our model — {label} (`qualitative_report.md`)",
        max_chars=5000,
    )

viz = ART / "visualizations"
for name in [
    "di_overview.png",
    "severity_comparison.png",
    "german_credit_disparate_impact.png",
    "hmda_disparate_impact.png",
]:
    p = viz / name
    if p.exists():
        display(Image(filename=str(p)))

## 4. Semantic Scholar evidence (sample)

Queries are built in `scripts/scholarly_evidence.py` from dataset name, protected attributes, and deterministic causes/mitigations. Papers prefer `citation_count >= 5` when possible, with fallback for newer work.

In [ ]:
ev_path = ART / "german_credit" / "fairness" / "qualitative_research_evidence.json"
if ev_path.exists():
    data = json.loads(ev_path.read_text(encoding="utf-8"))
    for attr, papers in list(data.items())[:1]:
        print(f"Attribute: {attr}")
        for i, paper in enumerate(papers[:3], 1):
            print(f"  {i}. {paper.get('title', '')[:80]}...")
            print(f"     citations={paper.get('citation_count')}, meets_threshold={paper.get('meets_citation_threshold', 'n/a')}")
else:
    print("Run qualitative step to generate", ev_path)

## 4b. Scholarly evidence in the LLM benchmark (`research_grounding` subscore)

`scripts/scholarly_evidence.py` fires up to 6 Semantic Scholar queries per attribute and injects retrieved papers into the LLM prompt. The scoring oracle awards **10 pts** if the response cites at least one real paper, **0 pts** otherwise.

During our gpt-4o run, Semantic Scholar returned **403 Forbidden** for all queries (rate-limit with no API key), so the injected evidence block was empty. Two patterns emerge:

- **Cycles 1–3** (zero-shot / chain-of-thought / self-critique): the LLM is not explicitly required to cite — most attributes score 0/10, with a few exceptions where gpt-4o proactively cites from training memory.
- **Cycle 4** (constrained): the system prompt explicitly requires citations, so gpt-4o self-cites from training knowledge every time — consistently 10/10.

The heatmap below shows `research_grounding` (0 or 10) per attribute × cycle for both datasets.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

BENCH = ART.parent / "artifacts" / "llm_benchmark" / "gpt-4o"
# ART already points to artifacts/, so:
BENCH = ART / ".." / "artifacts" / "llm_benchmark" / "gpt-4o"
BENCH = (ART / "llm_benchmark" / "gpt-4o").resolve()

datasets = [
    ("german_credit", "German Credit"),
    ("hmda", "HMDA Georgia"),
]
cycle_labels = ["C1\nzero-shot", "C2\nchain-of-thought", "C3\nself-critique", "C4\nconstrained"]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
cmap = mcolors.ListedColormap(["#d9534f", "#5cb85c"])
norm = mcolors.BoundaryNorm([0, 5, 10.1], cmap.N)

for ax, (ds_key, ds_label) in zip(axes, datasets):
    ds_dir = BENCH / ds_key
    if not ds_dir.exists():
        ax.set_title(f"{ds_label}\n(no artifacts)")
        continue

    rows = {}
    for f in sorted(ds_dir.glob("*.json")):
        if f.name == "_summary.json":
            continue
        d = json.loads(f.read_text())
        attr = d.get("attribute", f.stem.rsplit("_cycle", 1)[0])
        cycle = int(d.get("cycle", 0))
        rg = d.get("score", {}).get("subscores", {}).get("research_grounding", 0)
        rows.setdefault(attr, {})[cycle] = rg

    attrs = sorted(rows)
    matrix = np.array([[rows[a].get(c, 0) for c in range(1, 5)] for a in attrs])

    im = ax.imshow(matrix, cmap=cmap, norm=norm, aspect="auto")
    ax.set_xticks(range(4))
    ax.set_xticklabels(cycle_labels, fontsize=8)
    ax.set_yticks(range(len(attrs)))
    ax.set_yticklabels(attrs, fontsize=8)
    ax.set_title(f"{ds_label}", fontsize=10, fontweight="bold")

    for i, attr in enumerate(attrs):
        for j, cycle in enumerate(range(1, 5)):
            val = int(rows[attr].get(cycle, 0))
            ax.text(j, i, str(val), ha="center", va="center", fontsize=9,
                    color="white" if val == 0 else "black", fontweight="bold")

cbar = fig.colorbar(im, ax=axes, orientation="vertical", fraction=0.03, pad=0.04)
cbar.set_ticks([2.5, 7.5])
cbar.set_ticklabels(["0 — no citation", "10 — cited"])
fig.suptitle("research_grounding subscore (0 or 10) per attribute × cycle  |  gpt-4o", fontsize=11)
plt.tight_layout()
plt.show()

**Finding:** Cycle 4 (constrained) achieves 10/10 for every attribute across both datasets because the prompt requires in-text citations. Earlier cycles rely on gpt-4o's initiative: `foreign_worker` and `race` pick up sporadic citations from Cycles 2–3 (likely because those attributes appear in well-known bias literature the model has internalized), while other attributes score 0 unless forced.

This exposes a limitation: 16 out of 24 research-grounding scores reflect self-citation from model weights, not verified retrieved evidence. Adding a funded Semantic Scholar API key would replace training-data recall with grounded retrieval and remove this confound from the benchmark.

## 5. LLM benchmark runs — OpenAI & Gemini

Saved runs under `artifacts/{dataset}/fairness/openai/` and `.../gemini/`. Python already computed metrics; the LLM only does qualitative reasoning + refinement. **No API calls** in this cell.

In [ ]:
rows = []
for ds, label in [("german_credit", "German Credit"), ("hmda", "HMDA")]:
    for provider, folder in [("OpenAI", "openai"), ("Gemini", "gemini")]:
        nap = ART / ds / "fairness" / folder / "llm_napkin_math.json"
        raw = ART / ds / "fairness" / folder / "llm_raw_response.json"
        if not nap.exists() or not raw.exists():
            rows.append(
                {
                    "dataset": label,
                    "provider": provider,
                    "status": "missing artifacts (run llm_benchmark)",
                }
            )
            continue
        nap_data = json.loads(nap.read_text(encoding="utf-8"))
        u = nap_data.get("usage", nap_data)
        r = json.loads(raw.read_text(encoding="utf-8"))
        cycles = r.get("cycles", [])
        final = cycles[-1]["score"]["total_score"] if cycles else None
        rows.append(
            {
                "dataset": label,
                "provider": provider,
                "final_score": final,
                "cost_usd": u.get("total_cost_usd"),
                "wall_clock_s": u.get("wall_clock_s"),
                "input_tokens": u.get("input_tokens"),
                "output_tokens": u.get("output_tokens"),
            }
        )

display(pd.DataFrame(rows))

### 5b. OpenAI qualitative report (excerpt)

From `artifacts/german_credit/fairness/openai/llm_fairness_report.md` — same deterministic metrics, LLM-written narrative and mitigations.

In [ ]:
show_markdown_excerpt(
    ART / "german_credit" / "fairness" / "openai" / "llm_fairness_report.md",
    "OpenAI — German Credit",
    max_chars=4000,
)

### 5c. Gemini qualitative report (excerpt)

From `artifacts/german_credit/fairness/gemini/llm_fairness_report.md`.

In [ ]:
show_markdown_excerpt(
    ART / "german_credit" / "fairness" / "gemini" / "llm_fairness_report.md",
    "Gemini — German Credit",
    max_chars=4000,
)

### 5d. Side-by-side benchmark narrative (German Credit)

`openai/benchmark_comparison.md` contrasts the deterministic baseline with OpenAI on the same run.

In [ ]:
show_markdown_excerpt(
    ART / "german_credit" / "fairness" / "openai" / "benchmark_comparison.md",
    "OpenAI benchmark comparison (deterministic vs OpenAI)",
    max_chars=3500,
)
if (ART / "german_credit" / "fairness" / "gemini" / "benchmark_comparison.md").exists():
    show_markdown_excerpt(
        ART / "german_credit" / "fairness" / "gemini" / "benchmark_comparison.md",
        "Gemini benchmark comparison",
        max_chars=2500,
    )

## 6. Visuals (`artifacts/visualizations/`)

Benchmark and fairness plots: OpenAI cycle/agreement/cost charts, consolidated severity comparison, and dataset DI overviews.

In [ ]:
viz = ART / "visualizations"
for name in [
    "openai_cycle_scores.png",
    "openai_severity_agreement.png",
    "openai_cost_latency.png",
    "openai_di_context.png",
    "severity_comparison.png",
    "delta_heatmap.png",
    "cost_summary.png",
]:
    p = viz / name
    if p.exists():
        display(Image(filename=str(p)))

## 7. Deep dive — full file paths

| System | Location |
|--------|----------|
| Our model (deterministic) | `artifacts/{dataset}/fairness/qualitative_report.md` |
| OpenAI | `artifacts/{dataset}/fairness/openai/llm_fairness_report.md`, `benchmark_comparison.md` |
| Gemini | `artifacts/{dataset}/fairness/gemini/llm_fairness_report.md`, `benchmark_comparison.md` |
| Consolidated comparison | `artifacts/consolidated/deterministic_vs_openai_short_report.pdf` |

## 8. Re-run pipeline (optional)

```bash
./venv/bin/python run_pipeline.py --datasets german_credit hmda
```

Then re-run all cells to refresh excerpts and figures.